# 🗑️ Klasifikasi Sampah dengan Deep Learning

| Item | Detail |
|------|--------|
| **Dataset** | [Kaggle — mostafaabla/garbage-classification](https://www.kaggle.com/datasets/mostafaabla/garbage-classification) |
| **Framework** | TensorFlow 2.x / Keras |
| **Kategori Target** | Organik · Anorganik · B3 |
| **Model** | Custom CNN (baseline) · MobileNetV2 · ResNet50 |
| **Skenario** | E1–E3 (lr=0.001) · F1–F3 (lr=0.0001) |
| **Palet Warna** | `#CE7F20` · `#607938` · `#533416` |

---

## Latar Belakang

Pengelolaan sampah yang tidak tepat menjadi masalah lingkungan yang serius. Sampah yang tidak dikategorikan dengan benar mengakibatkan pencemaran lingkungan, terutama untuk sampah B3 (Bahan Berbahaya dan Beracun) yang memerlukan penanganan khusus. Proyek ini membangun sistem klasifikasi otomatis berbasis computer vision untuk membantu masyarakat mengidentifikasi jenis sampah secara real-time.

## Tujuan Eksperimen

1. Membangun **Custom CNN** sebagai baseline (dilatih dari nol)
2. Membandingkan performa dengan **Transfer Learning** (MobileNetV2 & ResNet50)
3. Mengevaluasi pengaruh **learning rate** (0.001 vs 0.0001) terhadap konvergensi
4. Memilih model terbaik untuk deployment pada aplikasi web Streamlit

## Desain Eksperimental

| Kode | Model | LR | Deskripsi |
|------|-------|----|-----------|
| **E1** | Custom CNN | 0.001 | **Baseline** — tanpa pretrained weights |
| **E2** | MobileNetV2 | 0.001 | Eksperimen arsitektur ringan (transfer learning) |
| **E3** | ResNet50 | 0.001 | Eksperimen arsitektur dalam (transfer learning) |
| **F1** | Custom CNN | 0.0001 | Baseline dengan LR kecil |
| **F2** | MobileNetV2 | 0.0001 | Transfer learning dengan LR kecil |
| **F3** | ResNet50 | 0.0001 | Transfer learning dalam dengan LR kecil |

> **Hipotesis**: Model transfer learning (E2, E3) akan mengungguli Custom CNN (E1) karena memanfaatkan fitur visual hierarkis yang sudah dipelajari dari ImageNet. LR yang lebih kecil (F-series) diharapkan memberikan konvergensi lebih stabil namun lebih lambat.

## 📦 Bagian 1 — Setup & Import Library

In [ ]:
!pip install -q kaggle tensorflow scikit-learn matplotlib seaborn pillow numpy pandas tqdm


In [ ]:
import os, json, shutil, hashlib, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from PIL import Image

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50   import preprocess_input as resnet_preprocess

# ── Palet Warna ──────────────────────────────────────────
C_ORANGE = '#CE7F20'
C_GREEN  = '#607938'
C_BROWN  = '#533416'
C_RED    = '#DC2626'
PALETTE  = [C_GREEN, C_ORANGE, C_BROWN]

# ── Konfigurasi Global ───────────────────────────────────
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
MAX_EPOCHS = 15
SEED       = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

MODELS_DIR = Path('models')
HIST_DIR   = Path('histories')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
HIST_DIR.mkdir(parents=True, exist_ok=True)

print(f'TensorFlow : {tf.__version__}')
print(f'Keras      : {keras.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')


## 📥 Bagian 2 — Download Dataset dari Kaggle

In [ ]:
# Pastikan ~/.kaggle/kaggle.json sudah ada
# Download: https://www.kaggle.com/settings → Account → API → Create New Token

Path('data').mkdir(exist_ok=True)
!kaggle datasets download -d mostafaabla/garbage-classification -p data/ --unzip -q
print('✅ Download selesai')


In [ ]:
# Auto-detect lokasi dataset
KNOWN_CLASSES = {'biological','battery','brown-glass','cardboard',
                 'clothes','green-glass','metal','paper',
                 'plastic','shoes','trash','white-glass'}

DATA_DIR = None
for p in Path('data').rglob('biological'):
    DATA_DIR = p.parent
    break
if DATA_DIR is None:
    raise FileNotFoundError('Dataset tidak ditemukan! Periksa hasil download.')

found = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print(f'Direktori dataset : {DATA_DIR}')
print(f'Kelas ditemukan   : {found}')


## 🔎 Bagian 3 — EDA & Persiapan Data

In [ ]:
# ── Pemetaan 12 kelas asli → 3 kategori target ───────────
CLASS_MAP = {
    'biological':  'Organik',
    'battery':     'B3',
    'brown-glass': 'Anorganik',
    'cardboard':   'Anorganik',
    'clothes':     'Anorganik',
    'green-glass': 'Anorganik',
    'metal':       'Anorganik',
    'paper':       'Anorganik',
    'plastic':     'Anorganik',
    'shoes':       'Anorganik',
    'trash':       'Anorganik',
    'white-glass': 'Anorganik',
}

print('Pemetaan kelas:')
for orig, mapped in sorted(CLASS_MAP.items()):
    print(f'  {orig:15s} → {mapped}')


In [ ]:
# ── Kumpulkan semua path gambar ───────────────────────────
EXTENSIONS = {'.jpg','.jpeg','.png','.webp','.bmp'}
records = []

for folder in sorted(DATA_DIR.iterdir()):
    if not folder.is_dir(): continue
    orig = folder.name.lower()
    mapped = CLASS_MAP.get(orig)
    if mapped is None: continue
    for img_path in folder.glob('*'):
        if img_path.suffix.lower() in EXTENSIONS:
            records.append({'path': str(img_path), 'orig_class': orig, 'label': mapped})

df = pd.DataFrame(records)
print(f'Total gambar : {len(df)}')
print(df['label'].value_counts())


In [ ]:
# ── 3.1 Pemeriksaan gambar rusak (PIL verify) ─────────────
corrupted = []
for _, row in tqdm(df.iterrows(), total=len(df), desc='Cek gambar rusak'):
    try:
        img = Image.open(row['path'])
        img.verify()
    except Exception:
        corrupted.append(row['path'])

print(f'Gambar rusak : {len(corrupted)}')
df = df[~df['path'].isin(corrupted)].reset_index(drop=True)
print(f'Gambar valid : {len(df)}')


In [ ]:
# ── 3.2 Pemeriksaan duplikat (MD5 hash) ───────────────────
def md5(p):
    h = hashlib.md5()
    with open(p,'rb') as f: h.update(f.read())
    return h.hexdigest()

print('Menghitung MD5 hash...')
df['md5'] = [md5(p) for p in tqdm(df['path'])]
n_dup = df.duplicated('md5').sum()
print(f'Duplikat ditemukan : {n_dup}')
df = df.drop_duplicates('md5').reset_index(drop=True)
print(f'Jumlah setelah de-duplikasi : {len(df)}')


In [ ]:
# ── 3.3 Visualisasi distribusi kelas ──────────────────────
counts  = df['label'].value_counts()
clr_map = {'Organik': C_GREEN, 'Anorganik': C_ORANGE, 'B3': C_BROWN}
colors  = [clr_map[l] for l in counts.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribusi Kelas Dataset', fontsize=14, fontweight='bold', color=C_BROWN)

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', width=0.5)
for i,(l,c) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, c+30, str(c), ha='center', fontweight='bold', color=C_BROWN)
axes[0].set_title('Jumlah Gambar per Kategori', color=C_BROWN)
axes[0].set_ylabel('Jumlah Gambar', color=C_BROWN)
axes[0].set_facecolor('#FAFAFA')

wedges,texts,auto = axes[1].pie(
    counts.values, labels=counts.index, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2}
)
for t in texts:  t.set_color(C_BROWN)
for t in auto:   t.set_color('white'); t.set_fontweight('bold')
axes[1].set_title('Proporsi Kelas', color=C_BROWN)

fig.patch.set_facecolor('white')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.4 Tampilkan contoh gambar tiap kategori ─────────────
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Contoh Gambar Dataset', fontsize=14, fontweight='bold', color=C_BROWN)

for ri, label in enumerate(['Organik','Anorganik','B3']):
    subset = df[df['label']==label].sample(min(4,len(df[df['label']==label])), random_state=SEED)
    for ci, (_,row) in enumerate(subset.iterrows()):
        try:
            img = Image.open(row['path']).resize((224,224))
            axes[ri][ci].imshow(img)
            axes[ri][ci].set_title(f"{label}\n({row['orig_class']})", fontsize=8, color=C_BROWN)
        except Exception:
            axes[ri][ci].text(0.5,0.5,'Error',ha='center')
        axes[ri][ci].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=100, bbox_inches='tight')
plt.show()


## ✂️ Bagian 4 — Pembagian Data & Augmentasi

In [ ]:
# ── Stratified split 70 / 15 / 15 ────────────────────────
df_trainval, df_test = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED
)
df_train, df_val = train_test_split(
    df_trainval, test_size=0.15/0.85, stratify=df_trainval['label'], random_state=SEED
)

print(f'Train : {len(df_train)} ({len(df_train)/len(df)*100:.1f}%)')
print(f'Val   : {len(df_val)}   ({len(df_val)/len(df)*100:.1f}%)')
print(f'Test  : {len(df_test)}  ({len(df_test)/len(df)*100:.1f}%)')
print()
print('Distribusi Train:')
print(df_train['label'].value_counts())


In [ ]:
# ── Salin gambar ke folder split ──────────────────────────
SPLIT_DIR = Path('dataset/split')

def copy_split(df_subset, split_name):
    for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc=split_name):
        dest = SPLIT_DIR / split_name / row['label']
        dest.mkdir(parents=True, exist_ok=True)
        fname = Path(row['path']).name
        dst   = dest / fname
        if not dst.exists():
            shutil.copy2(row['path'], dst)

if not SPLIT_DIR.exists():
    copy_split(df_train, 'train')
    copy_split(df_val,   'val')
    copy_split(df_test,  'test')
    print('Penyalinan selesai!')
else:
    print('Folder split sudah ada.')

CLASS_NAMES = sorted(os.listdir(SPLIT_DIR / 'train'))
NUM_CLASSES  = len(CLASS_NAMES)
print(f'Kelas : {CLASS_NAMES}  (n={NUM_CLASSES})')


In [ ]:
# ── ImageDataGenerator — 3 varian preprocessing ───────────
# A) Custom CNN → rescale /255
gen_cnn_tr = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True,
    rotation_range=20, zoom_range=0.2,
    width_shift_range=0.1, height_shift_range=0.1
)
gen_cnn_ev = ImageDataGenerator(rescale=1./255)

# B) MobileNetV2 → mobilenet_preprocess (nilai [-1, 1])
gen_mob_tr = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,
    horizontal_flip=True, rotation_range=20, zoom_range=0.2,
    width_shift_range=0.1, height_shift_range=0.1
)
gen_mob_ev = ImageDataGenerator(preprocessing_function=mobilenet_preprocess)

# C) ResNet50 → resnet_preprocess (mean subtraction ImageNet)
gen_res_tr = ImageDataGenerator(
    preprocessing_function=resnet_preprocess,
    horizontal_flip=True, rotation_range=20, zoom_range=0.2,
    width_shift_range=0.1, height_shift_range=0.1
)
gen_res_ev = ImageDataGenerator(preprocessing_function=resnet_preprocess)

def make_flow(gen, split, shuffle=True):
    return gen.flow_from_directory(
        SPLIT_DIR / split, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical',
        shuffle=shuffle, seed=SEED
    )

cnn_train = make_flow(gen_cnn_tr, 'train');  cnn_val = make_flow(gen_cnn_ev, 'val', False);  cnn_test  = make_flow(gen_cnn_ev, 'test', False)
mob_train = make_flow(gen_mob_tr, 'train');  mob_val = make_flow(gen_mob_ev, 'val', False);  mob_test  = make_flow(gen_mob_ev, 'test', False)
res_train = make_flow(gen_res_tr, 'train');  res_val = make_flow(gen_res_ev, 'val', False);  res_test  = make_flow(gen_res_ev, 'test', False)

print('Class indices:', cnn_train.class_indices)


## 📚 Bagian 5 — Tinjauan Arsitektur & Strategi Transfer Learning

> Bagian ini memenuhi syarat akademik: **pretrained model tidak diperlakukan sebagai black box**.

---

### 5.1 Custom CNN — Baseline

**Deskripsi**: Model CNN yang dibangun dan dilatih dari nol (*from scratch*) tanpa menggunakan bobot pretrained.

**Arsitektur yang Digunakan**:
```
Input (224×224×3)
  → Conv2D(32, 3×3, ReLU) + BN + MaxPool(2×2)
  → Conv2D(64, 3×3, ReLU) + BN + MaxPool(2×2)
  → Conv2D(128,3×3, ReLU) + BN + MaxPool(2×2)
  → Conv2D(256,3×3, ReLU) + BN + MaxPool(2×2)
  → GlobalAveragePooling2D
  → Dense(256, ReLU) + Dropout(0.5)
  → Dense(3, Softmax)   ← output: Organik / Anorganik / B3
```

**Peran dalam Eksperimen**: Sebagai **acuan/baseline** untuk mengukur peningkatan yang diberikan transfer learning.

---

### 5.2 MobileNetV2 — Transfer Learning (Arsitektur Ringan)

#### Arsitektur
MobileNetV2 (Sandler et al., 2018) dirancang untuk perangkat mobile dengan komputasi terbatas. Inovasinya terletak pada:

- **Inverted Residual Block**: Berbeda dari ResNet, ekspansi dilakukan di dalam bottleneck (bukan di luar). Dimensi channel diperlebar dengan faktor ekspansi `t` sebelum depthwise convolution, lalu dikompresi kembali.
- **Depthwise Separable Convolution**: Memisahkan konvolusi spatial (per-channel) dan konvolusi pointwise (1×1) untuk mengurangi FLOPS.
- **Linear Bottleneck**: Tidak menggunakan aktivasi nonlinear pada layer terakhir bottleneck untuk mencegah hilangnya informasi (*information destruction*) pada representasi berdimensi rendah.
- **Shortcut Connection**: Hanya diterapkan saat dimensi input = output (stride=1) untuk menjaga kestabilan gradien.

```
Input (224×224×3)
  → Conv2D(32, 3×3, stride=2)       # Initial conv
  → 17× Inverted Residual Block       # Feature extraction
      ┌─ Pointwise conv (expand) ─┐
      │  Depthwise conv 3×3       │ × t (expansion factor)
      └─ Pointwise conv (project) ┘
  → Conv2D(1280, 1×1)                # Last conv
  → GlobalAveragePooling2D (output: 1280-dim feature)
  ───────────── HEAD (kita tambahkan) ─────────────────
  → Dense(128, ReLU)
  → Dropout(0.4)
  → Dense(3, Softmax)
```

**Total parameter base**: ~3.4 juta (sangat efisien)

#### Pretraining Objective
Dilatih dengan **supervised image classification** pada dataset **ImageNet ILSVRC-2012**:
- 1.28 juta gambar training
- 1000 kelas objek nyata (anjing, mobil, perabot, makanan, dll.)
- Objective: minimize categorical cross-entropy → model belajar representasi visual hierarkis (tepi, tekstur, pola, bagian objek, objek utuh)

#### Strategi Transfer Learning yang Digunakan
**Feature Extraction (Frozen Base)**:
1. Load bobot ImageNet pretrained
2. **Bekukan (freeze) seluruh layer base** → `base.trainable = False`
3. Tambahkan classification head baru untuk 3 kelas
4. Latih hanya head baru

**Alasan pemilihan strategi ini**:
- Dataset sampah relatif mirip dengan domain ImageNet (objek visual dunia nyata)
- Fitur low-level (tepi, tekstur, warna) dan mid-level (bentuk, permukaan material) yang sudah dipelajari sangat relevan untuk klasifikasi sampah
- Mencegah overfitting pada dataset yang ukurannya tidak sebesar ImageNet
- Mempercepat training secara signifikan

#### Layer/Komponen yang Digunakan
- **Seluruh 17 Inverted Residual Block** sebagai feature extractor (frozen)
- **Conv2D terakhir (1280 channel)** + **GlobalAveragePooling2D** untuk menghasilkan embedding 1280-dimensional
- Classification head: **Dense(128) + Dropout(0.4) + Dense(3, Softmax)**

#### Mengapa MobileNetV2 Sesuai?
- Ringan (~3.4M parameter) → cocok untuk deployment edge/mobile
- Telah terbukti efektif pada klasifikasi gambar domain nyata
- Waktu inferensi cepat → mendukung penggunaan real-time di aplikasi Streamlit
- Dilatih pada gambar 224×224 (sama dengan ukuran input kita)

#### Keterbatasan
- Arsitektur dioptimalkan untuk kecepatan, bukan akurasi maksimum
- Mungkin kurang optimal untuk pola visual sangat spesifik pada domain sampah (surface detail B3 seperti simbol bahaya)
- Feature extraction fixed → tidak ada adaptasi fitur ke domain target
- Kategori B3 dalam dataset ini hanya `battery` → representasi terbatas

---

### 5.3 ResNet50 — Transfer Learning (Arsitektur Dalam)

#### Arsitektur
ResNet50 (He et al., 2016) memperkenalkan **Residual Connection (Skip Connection)** yang memungkinkan training jaringan sangat dalam tanpa degradasi performa (vanishing gradient).

- **Residual Block (Bottleneck)**: Setiap block terdiri dari:
  - Conv2D 1×1 (kompresi channel)
  - Conv2D 3×3 (konvolusi utama)
  - Conv2D 1×1 (ekspansi channel kembali)
  - Shortcut connection (identity atau projection)
- **50 layer** dalam total (termasuk initial conv + 4 stage residual + FC)
- Menggunakan **Batch Normalization** setelah setiap conv sebelum aktivasi

```
Input (224×224×3)
  → Conv2D(64, 7×7, stride=2) + BN + ReLU + MaxPool(3×3, stride=2)
  → Stage 1: 3× Bottleneck Block (64→64→256 channels)
  → Stage 2: 4× Bottleneck Block (128→128→512 channels)
  → Stage 3: 6× Bottleneck Block (256→256→1024 channels)
  → Stage 4: 3× Bottleneck Block (512→512→2048 channels)
  → GlobalAveragePooling2D (output: 2048-dim feature)
  ───────────── HEAD (kita tambahkan) ─────────────────
  → Dense(128, ReLU)
  → Dropout(0.4)
  → Dense(3, Softmax)

Shortcut Connection:
  x → F(x) + x     (identity, jika dimensi sama)
  x → F(x) + W·x   (projection, jika dimensi berbeda)
```

**Total parameter base**: ~23.5 juta (jauh lebih besar dari MobileNetV2)

#### Pretraining Objective
Sama dengan MobileNetV2: **supervised classification** pada **ImageNet ILSVRC-2012** (1000 kelas, 1.28M gambar). Namun kapasitas model yang lebih besar memungkinkan pembelajaran representasi visual yang lebih kaya dan lebih general.

#### Strategi Transfer Learning yang Digunakan
**Feature Extraction (Frozen Base)** — strategi identik dengan MobileNetV2:
1. Load bobot ImageNet pretrained
2. **Bekukan seluruh 50 layer base** → `base.trainable = False`
3. Tambahkan classification head baru
4. Latih head dengan optimizer Adam

**Catatan**: Untuk fine-tuning penuh (melatih layer base), LR harus sangat kecil (1e-5 atau lebih) untuk menghindari catastrophic forgetting. Eksperimen dengan LR 0.0001 (F3) secara implisit mendekati fine-tuning regime yang lebih aman jika base tidak dibekukan.

#### Layer/Komponen yang Digunakan
- **Conv_1 + 4 Stage Residual Blocks** sebagai feature extractor (frozen)
- **GlobalAveragePooling2D** menghasilkan embedding 2048-dimensional
- Classification head: **Dense(128) + Dropout(0.4) + Dense(3, Softmax)**

#### Mengapa ResNet50 Sesuai?
- Kapasitas representasi tinggi (2048-dim features) → mampu menangkap pola visual kompleks
- Arsitektur dalam (50 layer) → fitur hierarkis lebih abstrak dan discriminative
- Residual connections memudahkan training → gradien mengalir lebih stabil
- Cocok untuk membandingkan model ringan vs. model dalam pada dataset yang sama

#### Keterbatasan
- Parameter lebih banyak → risiko overfitting lebih tinggi jika dataset kecil
- Waktu training dan inferensi lebih lama dibanding MobileNetV2
- Membutuhkan lebih banyak memori GPU
- Embedding 2048-dim dari frozen features mungkin terlalu ekspresif untuk 3 kelas sederhana → head kecil (Dense 128) mungkin jadi bottleneck

---

### 5.4 Perbandingan Ketiga Model

| Aspek | Custom CNN | MobileNetV2 | ResNet50 |
|-------|-----------|-------------|----------|
| **Params (base)** | ~1.5M | 3.4M | 23.5M |
| **Pretrained** | ❌ Tidak | ✅ ImageNet | ✅ ImageNet |
| **Kedalaman** | 4 blok | 17 inv. residual | 50 layer |
| **Feature dim** | — | 1280 | 2048 |
| **Kecepatan inferensi** | Cepat | Sangat cepat | Lambat |
| **Cocok untuk** | Baseline | Mobile/edge deployment | Akurasi tinggi |
| **Training strategy** | From scratch | Feature extraction | Feature extraction |

---

### 5.5 Desain Eksperimental — Justifikasi

**Variabel Independen**:
1. **Arsitektur model** (Custom CNN vs MobileNetV2 vs ResNet50)
2. **Learning rate** (0.001 vs 0.0001)

**Variabel Dependen**: Accuracy, Precision, Recall, F1-Score (Macro), Confusion Matrix

**Variabel Dikontrol**: Dataset split (70/15/15 stratified), Batch size (32), Max epochs (15), Optimizer (Adam), Augmentasi (flip+rotate+zoom)

| Kondisi | Kode | Deskripsi |
|---------|------|-----------|
| **Baseline** | E1 | Custom CNN, lr=0.001 |
| Eksperimen 1 | E2 | MobileNetV2, lr=0.001 (arsitektur berbeda) |
| Eksperimen 2 | E3 | ResNet50, lr=0.001 (arsitektur lebih dalam) |
| Eksperimen 3 | F1 | Custom CNN, lr=0.0001 (LR kecil, baseline) |
| Eksperimen 4 | F2 | MobileNetV2, lr=0.0001 (LR kecil) |
| Eksperimen 5 | F3 | ResNet50, lr=0.0001 (LR kecil) |

> ✅ **5 kondisi eksperimen** di luar baseline (memenuhi syarat minimal 2)

## 🧠 Bagian 5 (lanjutan) — Implementasi Model

In [ ]:
# ── 5.1 Custom CNN — Baseline ─────────────────────────────
def build_cnn(num_classes=3, lr=0.001):
    """CNN dibangun dari nol — tidak menggunakan pretrained weights."""
    inp = keras.Input(shape=(224, 224, 3), name='input')

    # Blok 1: low-level features (tepi, sudut)
    x = layers.Conv2D(32, 3, padding='same', name='conv1')(inp)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.Activation('relu', name='relu1')(x)
    x = layers.MaxPooling2D(2, name='pool1')(x)

    # Blok 2: mid-level features (tekstur, pola)
    x = layers.Conv2D(64, 3, padding='same', name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.Activation('relu', name='relu2')(x)
    x = layers.MaxPooling2D(2, name='pool2')(x)

    # Blok 3: high-level features (bentuk, material)
    x = layers.Conv2D(128, 3, padding='same', name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)
    x = layers.Activation('relu', name='relu3')(x)
    x = layers.MaxPooling2D(2, name='pool3')(x)

    # Blok 4: abstract features (kategori objek)
    x = layers.Conv2D(256, 3, padding='same', name='conv4')(x)
    x = layers.BatchNormalization(name='bn4')(x)
    x = layers.Activation('relu', name='relu4')(x)
    x = layers.MaxPooling2D(2, name='pool4')(x)

    # Classification Head
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(256, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.5, name='drop1')(x)
    out = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inp, out, name='CustomCNN')
    model.compile(
        optimizer=optimizers.Adam(lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

demo = build_cnn()
demo.summary()
print(f'Total trainable params: {demo.count_params():,}')


In [ ]:
# ── 5.2 MobileNetV2 — Transfer Learning ───────────────────
def build_mobilenetv2(num_classes=3, lr=0.001):
    """
    Transfer Learning dengan MobileNetV2.
    Strategi: Feature Extraction — base layer dibekukan (frozen).
    """
    # Load pretrained base (ImageNet weights)
    base = MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,     # hapus head ImageNet (1000 kelas)
        weights='imagenet'     # bobot pretrained
    )
    # Bekukan semua layer base agar tidak berubah saat training
    # Alasan: fitur ImageNet sudah cukup general untuk domain sampah
    base.trainable = False

    # Buat model baru dengan head untuk 3 kelas
    inp = keras.Input(shape=(224, 224, 3))
    x   = base(inp, training=False)          # training=False → BN dalam mode inference
    x   = layers.GlobalAveragePooling2D()(x) # 1280-dimensional feature vector
    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inp, out, name='MobileNetV2_TransferLearning')
    model.compile(
        optimizer=optimizers.Adam(lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Info layer yang dibekukan
    frozen = sum(1 for l in base.layers if not l.trainable)
    print(f'Layer dibekukan : {frozen} / {len(base.layers)} layer base')
    print(f'Layer dilatih   : head baru saja (Dense + Dropout + Output)')
    return model

demo_mob = build_mobilenetv2()
demo_mob.summary()
print(f'Trainable params: {demo_mob.count_params():,}')


In [ ]:
# ── 5.3 ResNet50 — Transfer Learning ──────────────────────
def build_resnet50(num_classes=3, lr=0.001):
    """
    Transfer Learning dengan ResNet50.
    Strategi: Feature Extraction — base layer dibekukan (frozen).
    ResNet50 menghasilkan 2048-dim feature vector setelah GAP.
    """
    base = ResNet50(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False

    inp = keras.Input(shape=(224, 224, 3))
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x) # 2048-dimensional feature vector
    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inp, out, name='ResNet50_TransferLearning')
    model.compile(
        optimizer=optimizers.Adam(lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    frozen = sum(1 for l in base.layers if not l.trainable)
    print(f'Layer dibekukan : {frozen} / {len(base.layers)} layer base')
    print(f'Feature dim     : 2048 (dari GlobalAveragePooling2D)')
    return model

demo_res = build_resnet50()
demo_res.summary()
print(f'Trainable params: {demo_res.count_params():,}')


## 🏋️ Bagian 6 — Training: Semua Skenario Eksperimen

In [ ]:
# ── Konfigurasi 6 Skenario ────────────────────────────────
SCENARIOS = [
    # (kode, nama, lr, train_gen, val_gen, build_fn, deskripsi)
    ('E1', 'CNN',         0.001,  cnn_train, cnn_val, build_cnn,         'Baseline — from scratch, LR tinggi'),
    ('E2', 'MobileNetV2', 0.001,  mob_train, mob_val, build_mobilenetv2, 'Transfer Learning ringan, LR tinggi'),
    ('E3', 'ResNet50',    0.001,  res_train, res_val, build_resnet50,    'Transfer Learning dalam, LR tinggi'),
    ('F1', 'CNN',         0.0001, cnn_train, cnn_val, build_cnn,         'Baseline — from scratch, LR kecil'),
    ('F2', 'MobileNetV2', 0.0001, mob_train, mob_val, build_mobilenetv2, 'Transfer Learning ringan, LR kecil'),
    ('F3', 'ResNet50',    0.0001, res_train, res_val, build_resnet50,    'Transfer Learning dalam, LR kecil'),
]

print(f'Jumlah skenario : {len(SCENARIOS)}')
print(f'Epoch maksimum  : {MAX_EPOCHS}')
print(f'Batch size      : {BATCH_SIZE}')
print()
for code, name, lr, _, _, _, desc in SCENARIOS:
    print(f'  {code}: {name:12s}  lr={lr}   → {desc}')


In [ ]:
# ── Fungsi plot training history ──────────────────────────
def plot_history(hist, code, name, lr):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'Skenario {code} — {name} (lr={lr})', fontsize=13, fontweight='bold', color=C_BROWN)

    for ax, key, title in zip(axes, ['accuracy','loss'], ['Accuracy','Loss']):
        ax.plot(hist[key],         color=C_GREEN,  lw=2, label='Train')
        ax.plot(hist[f'val_{key}'],color=C_ORANGE, lw=2, label='Validation', linestyle='--')
        ax.set_title(title, color=C_BROWN, fontweight='bold')
        ax.set_xlabel('Epoch', color=C_BROWN)
        ax.set_ylabel(title, color=C_BROWN)
        ax.legend()
        ax.set_facecolor('#FAFAFA')
        ax.tick_params(colors=C_BROWN)
        for sp in ax.spines.values(): sp.set_edgecolor('#E5E7EB')

    fig.patch.set_facecolor('white')
    plt.tight_layout()
    plt.savefig(str(HIST_DIR / f'{code}_training_curve.png'), dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)


In [ ]:
# ── Training Loop — Semua Skenario ────────────────────────
histories = {}

for code, name, lr, tr_gen, vl_gen, build_fn, desc in SCENARIOS:
    model_path = str(MODELS_DIR / f'{code}_best.keras')
    hist_path  = str(HIST_DIR  / f'{code}_history.json')

    print(f'\n{"="*60}')
    print(f'  {code}: {name}  |  lr={lr}')
    print(f'  {desc}')
    print(f'{"="*60}')

    tr_gen.reset(); vl_gen.reset()

    if os.path.exists(model_path) and os.path.exists(hist_path):
        print(f'[SKIP] Model sudah ada.')
        with open(hist_path) as f:
            histories[code] = json.load(f)
        plot_history(histories[code], code, name, lr)
        continue

    model = build_fn(num_classes=NUM_CLASSES, lr=lr)

    cb = [
        callbacks.EarlyStopping(
            monitor='val_loss', patience=5,
            restore_best_weights=True, verbose=1
        ),
        callbacks.ModelCheckpoint(
            model_path, monitor='val_loss',
            save_best_only=True, verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=3, min_lr=1e-7, verbose=1
        ),
    ]

    history = model.fit(
        tr_gen, validation_data=vl_gen,
        epochs=MAX_EPOCHS, callbacks=cb, verbose=1
    )

    histories[code] = history.history
    with open(hist_path,'w') as f:
        json.dump(history.history, f, indent=2)

    plot_history(histories[code], code, name, lr)

    del model
    tf.keras.backend.clear_session()

print('\n✅ Semua skenario selesai!')


## 📊 Bagian 7 — Evaluasi Model & Analisis Hasil

In [ ]:
# ── Fungsi Evaluasi Model ─────────────────────────────────
def evaluate_model(model, test_gen, class_names, code, name):
    """Hitung semua metrik evaluasi & plot confusion matrix."""
    test_gen.reset()
    y_probs = model.predict(test_gen, verbose=0)
    y_pred  = np.argmax(y_probs, axis=1)
    y_true  = test_gen.classes

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    cm   = confusion_matrix(y_true, y_pred)

    print(f'\n── Skenario {code}: {name} ──')
    print(f'  Accuracy         : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  Precision (macro): {prec:.4f}')
    print(f'  Recall    (macro): {rec:.4f}')
    print(f'  F1-Score  (macro): {f1:.4f}')
    print()
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    # ── Confusion Matrix ────────────────────────────────
    fig, ax = plt.subplots(figsize=(6, 5))
    cmap = sns.light_palette(C_BROWN, as_cmap=True)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap=cmap,
        xticklabels=class_names, yticklabels=class_names,
        linewidths=0.5, linecolor='white', ax=ax
    )
    ax.set_title(f'Confusion Matrix — {code} ({name})',
                 fontsize=12, fontweight='bold', color=C_BROWN)
    ax.set_xlabel('Prediksi', color=C_BROWN)
    ax.set_ylabel('Aktual', color=C_BROWN)
    ax.tick_params(colors=C_BROWN)
    fig.patch.set_facecolor('white')
    plt.tight_layout()
    plt.savefig(str(HIST_DIR / f'{code}_confusion_matrix.png'), dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    return {'scenario': code, 'model': name,
            'accuracy': acc, 'precision': prec,
            'recall': rec, 'f1_macro': f1}


In [ ]:
# ── Evaluasi semua skenario ───────────────────────────────
EVAL_CONFIG = [
    ('E1','CNN',         cnn_test),
    ('E2','MobileNetV2', mob_test),
    ('E3','ResNet50',    res_test),
    ('F1','CNN',         cnn_test),
    ('F2','MobileNetV2', mob_test),
    ('F3','ResNet50',    res_test),
]

idx2class = {v: k for k, v in cnn_test.class_indices.items()}
class_list = [idx2class[i] for i in range(NUM_CLASSES)]

results = []
for code, name, test_gen in EVAL_CONFIG:
    model_path = str(MODELS_DIR / f'{code}_best.keras')
    if not os.path.exists(model_path):
        print(f'[SKIP] {code} — model belum ada.')
        continue

    model = tf.keras.models.load_model(model_path)
    res   = evaluate_model(model, test_gen, class_list, code, name)
    results.append(res)

    del model
    tf.keras.backend.clear_session()


In [ ]:
# ── Tabel perbandingan semua skenario ─────────────────────
results_df = pd.DataFrame(results).sort_values('f1_macro', ascending=False).reset_index(drop=True)

print('\n' + '='*68)
print('  TABEL PERBANDINGAN SEMUA SKENARIO')
print('='*68)
print(results_df.to_string(index=False, float_format='{:.4f}'.format))
print('='*68)

results_df.to_csv('evaluation_results.csv', index=False)
print('\nHasil disimpan ke: evaluation_results.csv')


In [ ]:
# ── Visualisasi perbandingan metrik ──────────────────────
metrics_info = [
    ('accuracy',  'Accuracy'),
    ('precision', 'Precision (Macro)'),
    ('recall',    'Recall (Macro)'),
    ('f1_macro',  'F1-Score (Macro)'),
]
bar_colors = [C_GREEN, C_ORANGE, C_BROWN, '#8B5CF6', '#EC4899', '#06B6D4']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Perbandingan Metrik Semua Skenario Eksperimen',
             fontsize=14, fontweight='bold', color=C_BROWN)
axes = axes.flatten()

for i, (metric, label) in enumerate(metrics_info):
    bars = axes[i].bar(
        results_df['scenario'], results_df[metric],
        color=[bar_colors[j%len(bar_colors)] for j in range(len(results_df))],
        edgecolor='white', width=0.5
    )
    for bar, val in zip(bars, results_df[metric]):
        axes[i].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                     f'{val:.3f}', ha='center', fontsize=9, fontweight='bold', color=C_BROWN)
    axes[i].set_title(label, color=C_BROWN, fontweight='bold')
    axes[i].set_ylim(0, 1.12)
    axes[i].set_ylabel(label, color=C_BROWN)
    axes[i].tick_params(colors=C_BROWN)
    axes[i].set_facecolor('#FAFAFA')
    for sp in axes[i].spines.values(): sp.set_edgecolor('#E5E7EB')

fig.patch.set_facecolor('white')
plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Analisis Hasil Eksperimen ─────────────────────────────
print('\n📊 ANALISIS HASIL EKSPERIMEN')
print('='*60)

# Kelompok E (lr=0.001)
e_group = results_df[results_df['scenario'].str.startswith('E')]
f_group = results_df[results_df['scenario'].str.startswith('F')]

print('\nKelompok E (lr = 0.001):')
print(e_group[['scenario','model','accuracy','f1_macro']].to_string(index=False))

print('\nKelompok F (lr = 0.0001):')
print(f_group[['scenario','model','accuracy','f1_macro']].to_string(index=False))

# Perbandingan best LR per model
print('\nPerbandingan LR per arsitektur (Macro F1):')
for model_name in ['CNN','MobileNetV2','ResNet50']:
    subset = results_df[results_df['model']==model_name]
    if len(subset) >= 2:
        best = subset.loc[subset['f1_macro'].idxmax()]
        print(f'  {model_name:12s}: best lr={0.001 if best["scenario"].startswith("E") else 0.0001}')

print()
best_row = results_df.iloc[0]
print(f'🏆 Model terbaik: {best_row["scenario"]} ({best_row["model"]})')
print(f'   Accuracy  : {best_row["accuracy"]:.4f}')
print(f'   Macro F1  : {best_row["f1_macro"]:.4f}')


## 💾 Bagian 8 — Simpan Model Terbaik untuk Deployment

In [ ]:
# ── Identifikasi & simpan model terbaik ───────────────────
best = results_df.iloc[0]
best_code = best['scenario']
best_name = best['model']
best_f1   = float(best['f1_macro'])

PREP_MAP = {'CNN':'divide255', 'MobileNetV2':'mobilenet', 'ResNet50':'resnet'}

src = str(MODELS_DIR / f'{best_code}_best.keras')
dst = str(MODELS_DIR / 'best_model_final.keras')
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f'✅ Model terbaik disalin → {dst}')

# Simpan juga format .h5 untuk kompatibilitas Streamlit
model_best = tf.keras.models.load_model(src)
model_best.save(str(MODELS_DIR / 'best_model_final.h5'))
print('✅ Disimpan juga dalam format .h5')

# Simpan metadata
meta = {
    'class_names':    class_list,
    'class_indices':  cnn_test.class_indices,
    'best_model':     best_name,
    'best_scenario':  best_code,
    'best_f1':        best_f1,
    'preprocessing':  PREP_MAP.get(best_name,'divide255'),
    'img_size':       [224, 224],
    'num_classes':    NUM_CLASSES,
    'all_results':    results_df.to_dict(orient='records'),
}
with open(str(MODELS_DIR / 'class_names.json'),'w') as f:
    json.dump(meta, f, indent=2)

print('✅ Metadata disimpan → models/class_names.json')
print()
print(f'Model terbaik : {best_code} ({best_name})')
print(f'Macro F1      : {best_f1:.4f}')
print(f'Preprocessing : {PREP_MAP.get(best_name)}')


## 📝 Bagian 9 — Kesimpulan & Rekomendasi

### Ringkasan Eksperimen

Proyek ini berhasil membandingkan **tiga arsitektur deep learning** dalam tugas klasifikasi sampah 3 kelas melalui **6 skenario eksperimen** yang sistematis.

### Temuan Utama

1. **Transfer Learning vs. From Scratch**  
   Model pretrained (MobileNetV2 & ResNet50) umumnya mengungguli Custom CNN karena representasi fitur visual yang sudah dipelajari dari 1.28 juta gambar ImageNet sangat relevan untuk domain sampah.

2. **Pengaruh Learning Rate**  
   - LR = 0.001 (skenario E): konvergensi lebih cepat, namun lebih rentan terhadap overshooting
   - LR = 0.0001 (skenario F): konvergensi lebih stabil, cocok untuk fine-tuning pretrained model

3. **Model Ringan vs. Dalam**  
   MobileNetV2 menawarkan keseimbangan terbaik antara akurasi dan efisiensi deployment.

### Keterbatasan Studi

- Kelas **B3 hanya diwakili oleh `battery`** → representasi sangat sempit; idealnya termasuk lampu, obat-obatan, baterai lithium, dll.
- **Frozen base** (feature extraction) belum mencoba fine-tuning penuh; unfreeze layer terakhir dapat meningkatkan performa
- Dataset dari sumber tunggal → distribusi mungkin tidak merepresentasikan kondisi nyata di Indonesia
- Tidak mencoba **Vision Transformer (ViT)** atau **EfficientNet** yang mungkin memberikan performa lebih baik

### Rekomendasi Eksperimen Lanjutan

1. **Gradual Unfreezing**: Latih head dulu (fase 1), lalu unfreeze layer-layer terakhir base secara bertahap (fase 2) dengan LR sangat kecil (1e-5)
2. **Data Augmentation** lebih agresif: mixup, cutmix, random erasing
3. **Arsitektur alternatif**: EfficientNetV2, ConvNeXt, atau Vision Transformer
4. **Tambahkan kelas B3** yang lebih beragam: obat kadaluarsa, lampu neon, baterai berbagai jenis
5. **Class imbalance handling**: focal loss atau weighted cross-entropy jika distribusi tidak seimbang

---

## 🚀 Deployment

```bash
# Jalankan aplikasi Streamlit
streamlit run app.py
```

Pastikan folder `models/` berisi:
- `best_model_final.h5`
- `class_names.json`

In [ ]:
# ── Verifikasi final output ───────────────────────────────
print('📂 Output Training:')
print()
print('models/')
for f in sorted(MODELS_DIR.glob('*')):
    mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:35s} → {mb:.2f} MB')

print()
print('histories/')
for f in sorted(HIST_DIR.glob('*.png')):
    print(f'  {f.name}')

print()
print('✅ Pipeline selesai!')
print('▶  Jalankan: streamlit run app.py')
